### Data Parsing & Quality Check

Loads the raw pulled data, builds a proper timestamp column, checks for
duplicates, documents nulls, and detects (but does not fill) missing hours
per sensor.


In [ ]:
import pandas as pd

RAW_PATH = "data/pedestrian_raw.parquet"
CLEAN_PATH = "data/pedestrian_clean.parquet"

## Load raw data

In [ ]:
df = pd.read_parquet(RAW_PATH)
print(f"Loaded {len(df)} rows")
df.head()

## Build timestamp column

Combines `sensing_date` (date string) and `hourday` (0-23) into a single
datetime column.

In [ ]:
df["timestamp"] = pd.to_datetime(df["sensing_date"]) + pd.to_timedelta(df["hourday"], unit="h")
df[["sensing_date", "hourday", "timestamp"]].head()

## Check for duplicate rows

In [ ]:
dup_by_id = df["id"].duplicated().sum()
dup_by_sensor_time = df.duplicated(subset=["location_id", "timestamp"]).sum()

print(f"Duplicate 'id' values: {dup_by_id}")
print(f"Duplicate (location_id, timestamp) pairs: {dup_by_sensor_time}")

## Document nulls in sensor_name / location

Not dropping these rows (decided earlier) — just recording counts per sensor
for the record.

In [ ]:
null_counts = (
    df[df["sensor_name"].isna()]
    .groupby("location_id")
    .size()
    .rename("rows_with_null_sensor_name")
)
print(null_counts if not null_counts.empty else "No null sensor_name rows found.")

## Confirm location_id maps to a fixed lat/lon

Each sensor (location_id) should correspond to exactly one physical
coordinate. This unnests the `location` column and checks that assumption
holds, rather than deriving anything new.

In [ ]:
df["lat"] = df["location"].apply(lambda loc: loc["lat"] if isinstance(loc, dict) else None)
df["lon"] = df["location"].apply(lambda loc: loc["lon"] if isinstance(loc, dict) else None)

coord_counts = df.groupby("location_id")[["lat", "lon"]].nunique()
inconsistent = coord_counts[(coord_counts["lat"] > 1) | (coord_counts["lon"] > 1)]

if inconsistent.empty:
    print("Each location_id maps to exactly one lat/lon pair.")
else:
    print("Inconsistent coordinates found for these location_id values:")
    print(inconsistent)

## Reference table: location_id to sensor_name / lat / lon

In [ ]:
location_lookup = (
    df.dropna(subset=["lat", "lon"])
    .drop_duplicates(subset=["location_id"])
    [["location_id", "sensor_name", "lat", "lon"]]
    .sort_values("location_id")
    .reset_index(drop=True)
)
location_lookup

## Detect missing hours per sensor

For each sensor, build the expected full hourly range between its earliest
and latest observed timestamp, and compare against what's actually present.
This only reports gaps -- no filling yet.

In [ ]:
gap_report = {}

for location_id, group in df.groupby("location_id"):
    observed = set(group["timestamp"])
    full_range = pd.date_range(
        start=group["timestamp"].min(),
        end=group["timestamp"].max(),
        freq="h",
    )
    missing = sorted(set(full_range) - observed)
    gap_report[location_id] = missing
    print(f"location_id={location_id}: {len(missing)} missing hours out of {len(full_range)} expected")

## Inspect missing timestamps for one sensor (optional)

Swap in a location_id from above to see the actual missing hours.

In [ ]:
example_location_id = list(gap_report.keys())[0]
gap_report[example_location_id][:20]  # first 20 missing timestamps, if any

## Sort and save parsed data

In [ ]:
df_parsed = df.sort_values(["location_id", "timestamp"]).reset_index(drop=True)

df_parsed.to_parquet(CLEAN_PATH, index=False)
print(f"Saved parsed data to {CLEAN_PATH} ({len(df_parsed)} rows)")